In [34]:
import duckdb as ddb

In [25]:
ddb.sql("install httpfs; load httpfs")

In [35]:
con = ddb.connect("../air_quality.db")

In [36]:
con.execute("create schema if not exists raw")

In [37]:
con.sql("""
    SET s3_access_key_id='';
    SET s3_secret_access_key='';
    SET s3_region='';
""")

In [38]:
con.execute("""
    create table if not exists raw.air_quality_data(
            location_id BIGINT,
            sensors_id BIGINT,
            "location" VARCHAR,
            "datetime" TIMESTAMP,
            lat DOUBLE,
            lon DOUBLE,
            "parameter" VARCHAR,
            units VARCHAR,
            "value" DOUBLE,
            "month" VARCHAR,
            "year" BIGINT,
            ingestion_datetime TIMESTAMP
    );
""")

In [68]:
con.execute("""
INSERT INTO raw.air_quality_data
SELECT 
    location_id, 
    sensors_id, 
    "location", 
    "datetime", 
    lat, 
    lon, 
    "parameter", 
    units, 
    "value",
    "month", 
    "year",
    current_timestamp AS ingestion_datetime
FROM read_csv('s3://openaq-data-archive/records/csv.gz/locationid=921005/year=2025/month=05/*.csv.gz');
""")

In [53]:
con.sql("select * from raw.air_quality_data where parameter in ('o3', 'no2', 'pm25', 'pm10','so2')")


┌─────────────┬────────────┬────────────────────────┬─────────────────────┬───────────────────┬─────────────────────┬───────────┬─────────┬────────┬─────────┬───────┬─────────────────────────┐
│ location_id │ sensors_id │        location        │      datetime       │        lat        │         lon         │ parameter │  units  │ value  │  month  │ year  │   ingestion_datetime    │
│    int64    │   int64    │        varchar         │      timestamp      │      double       │       double        │  varchar  │ varchar │ double │ varchar │ int64 │        timestamp        │
├─────────────┼────────────┼────────────────────────┼─────────────────────┼───────────────────┼─────────────────────┼───────────┼─────────┼────────┼─────────┼───────┼─────────────────────────┤
│        1528 │      22030 │ Vancouver Airport-1528 │ 2025-01-01 01:00:00 │ 49.18639000000001 │ -123.15221999999997 │ pm25      │ µg/m³   │    2.5 │ 01      │  2025 │ 2025-07-08 11:52:04.101 │
│        1528 │      22030 │ Vancou

In [69]:
con.sql("select count(*) from raw.air_quality_data")

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│        75128 │
└──────────────┘

In [70]:
con.close()